In [1]:
import pandas as pd
import joblib
import requests
import json

print("Libraries loaded!")


Libraries loaded!


In [2]:
rf_model = joblib.load('rf_model.pkl')
df = pd.read_csv('cleaned_data.csv')

FEATURE_COLS = [
    'signal_changed_mind', 'signal_high_discount', 'signal_late_return',
    'signal_bulk_return', 'signal_expensive_item', 'abuse_score',
    'Product_Price', 'Discount_Applied', 'Order_Quantity',
    'Days_to_Return', 'User_Age'
]

print("Model and data loaded!")

Model and data loaded!


In [4]:
def ask_ollama(prompt: str, model: str = "llama3.2") -> str:
    """
    Sends a prompt to local Ollama server and returns the response text.
    Make sure `ollama serve` is running in your terminal first.
    """
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False      # wait for full response before returning
        },
        timeout=60
    )
    response.raise_for_status()
    return response.json()["response"].strip()

In [6]:
def build_prompt(record: dict, prediction: int, probability: float) -> str:
    """
    Builds a structured prompt that gives the LLM all the context
    it needs to explain WHY this order is or isn't flagged as abuse.
    """
    
    # Which signals fired?
    fired = []
    if record['signal_changed_mind']   == 1: fired.append("returned with reason 'Changed mind'")
    if record['signal_high_discount']  == 1: fired.append(f"high discount applied ({record['Discount_Applied']}%) then returned")
    if record['signal_late_return']    == 1: fired.append(f"returned very late ({record['Days_to_Return']} days)")
    if record['signal_bulk_return']    == 1: fired.append(f"bulk order (qty {record['Order_Quantity']}) returned")
    if record['signal_expensive_item'] == 1: fired.append(f"expensive item (${record['Product_Price']}) returned")
    
    fired_text = "\n".join(f"  - {s}" for s in fired) if fired else "  - None"
    verdict    = "ABUSE DETECTED" if prediction == 1 else "LEGITIMATE RETURN"
    
    prompt = f"""You are a fraud analyst at an e-commerce company.
A machine learning model has reviewed a customer return request and made a decision.
Your job is to explain this decision clearly in 3-5 sentences for the fraud review team.

--- ORDER DETAILS ---
Product price:      ${record['Product_Price']}
Discount applied:   {record['Discount_Applied']}%
Order quantity:     {record['Order_Quantity']}
Days to return:     {record['Days_to_Return']}
Return reason:      {record.get('Return_Reason', 'N/A')}
Customer age:       {record['User_Age']}

--- ABUSE SIGNALS TRIGGERED ---
{fired_text}

--- MODEL DECISION ---
Verdict:     {verdict}
Confidence:  {probability:.1%}
Abuse score: {record['abuse_score']} out of 5 signals

--- YOUR TASK ---
Write a short, professional explanation of why this return was flagged (or not flagged).
Do not repeat the numbers mechanically. Explain the pattern and business risk clearly.
"""
    return prompt

print("Prompt builder ready!")

Prompt builder ready!


In [7]:
def predict_and_explain(record: dict) -> dict:
    """
    Takes a single order record dict, runs the RF model,
    then asks Ollama to explain the result.
    Returns a result dict with prediction, probability, and reasoning.
    """
    # Step 1: prepare features for model
    sample = pd.DataFrame([{col: record[col] for col in FEATURE_COLS}])
    
    # Step 2: get ML prediction
    prediction  = rf_model.predict(sample)[0]
    probability = rf_model.predict_proba(sample)[0][1]
    
    # Step 3: build prompt and get LLM reasoning
    prompt    = build_prompt(record, prediction, probability)
    reasoning = ask_ollama(prompt)
    
    return {
        "verdict":     "ABUSE" if prediction == 1 else "NOT ABUSE",
        "probability": f"{probability:.1%}",
        "abuse_score": record['abuse_score'],
        "reasoning":   reasoning
    }

print("Pipeline ready!")

Pipeline ready!


In [8]:
high_risk_order = {
    'signal_changed_mind':   1,
    'signal_high_discount':  1,
    'signal_late_return':    0,
    'signal_bulk_return':    0,
    'signal_expensive_item': 1,
    'abuse_score':           3,
    'Product_Price':         420.0,
    'Discount_Applied':      42.0,
    'Order_Quantity':        2,
    'Days_to_Return':        20.0,
    'User_Age':              29,
    'Return_Reason':         'Changed mind'
}

result = predict_and_explain(high_risk_order)

print(f"Verdict:     {result['verdict']}")
print(f"Confidence:  {result['probability']}")
print(f"Abuse score: {result['abuse_score']}/5")
print(f"\n--- LLM Reasoning ---")
print(result['reasoning'])

Verdict:     ABUSE
Confidence:  100.0%
Abuse score: 3/5

--- LLM Reasoning ---
This return has been flagged for potential abuse due to an unusual combination of factors. The customer returned an item with a significant discount applied and changed their mind as the reason, which is a red flag in itself. Additionally, the expensive product was returned on a relatively short days-to-return period, suggesting that the customer may have purchased the item impulsively or for resale purposes. This pattern raises business risk, particularly given the high discount applied and short return period.


In [9]:
low_risk_order = {
    'signal_changed_mind':   0,
    'signal_high_discount':  0,
    'signal_late_return':    0,
    'signal_bulk_return':    0,
    'signal_expensive_item': 0,
    'abuse_score':           0,
    'Product_Price':         49.99,
    'Discount_Applied':      10.0,
    'Order_Quantity':        1,
    'Days_to_Return':        5.0,
    'User_Age':              45,
    'Return_Reason':         'Defective'
}

result = predict_and_explain(low_risk_order)

print(f"Verdict:     {result['verdict']}")
print(f"Confidence:  {result['probability']}")
print(f"Abuse score: {result['abuse_score']}/5")
print(f"\n--- LLM Reasoning ---")
print(result['reasoning'])

Verdict:     NOT ABUSE
Confidence:  0.0%
Abuse score: 0/5

--- LLM Reasoning ---
This customer's return request is deemed legitimate due to a low confidence score from the machine learning model, indicating that it is unlikely that the product is defective and actually received as described. The absence of any abuse signals suggests that this case does not present a significant business risk. As such, we will approve the return, allowing the customer to receive a refund for the defective item.


In [10]:
# Pick row 0 from your actual data and run it through the full pipeline
real_row = df.iloc[0].to_dict()

result = predict_and_explain(real_row)

print(f"Verdict:     {result['verdict']}")
print(f"Confidence:  {result['probability']}")
print(f"Abuse score: {result['abuse_score']}/5")
print(f"\n--- LLM Reasoning ---")
print(result['reasoning'])

Verdict:     ABUSE
Confidence:  100.0%
Abuse score: 4/5

--- LLM Reasoning ---
Based on our review, this customer's return is flagged due to an unusual combination of factors that suggest potential abuse. The high discount applied ($45.27) followed by a late return (387 days) is concerning, as it may indicate that the order was placed for resale or not intended for genuine use. Additionally, the expensive item being returned is also red flags, suggesting possible price gouging or manipulation of our system. Our model has flagged this transaction with an abuse score of 4 out of 5 signals, indicating a high level of confidence in its decision.
